In [1]:
import numpy as np


from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import CXGate, PauliEvolutionGate
from qiskit.transpiler import PassManager, Layout
from qiskit.transpiler.passes import InverseCancellation, CommutativeCancellation
from qiskit.circuit import Parameter

from qiskit_aer import AerSimulator

from qopt_best_practices.transpilation.swap_cancellation_pass import SwapToFinalMapping

from hubo_qaoa.utils.get_swap_strategy import get_swap_strategy

from hubo_qaoa.utils.graph_to_hubo_hamiltonian import graph_to_hubo_hamiltonian
from hubo_qaoa.utils.gfa_utils import gfa_file_to_graph
from hubo_qaoa.utils.str_utils import genbin
from hubo_qaoa.utils.parameterise_circuit import parameterise_circuit

from qiskit_qaoa.utils.transpiler_passes import FindCommutingPauliEvolutionsMulti
from qiskit_qaoa.utils.commuting_gate_router_precompute_rzz import CommutingGateRouterPrecomputeRzz
from qiskit_qaoa.utils.commuting_gate_router_precompute import CommutingGateRouterPrecompute as CommutingGateRouterPrecomputeRz
from qiskit_qaoa.utils.commuting_gate_router_rzz import CommutingGateRouterRzz as CommutingGateRouterRzz
from qiskit_qaoa.utils.commuting_gate_router_all_to_all import CommutingGateRouterAllToAll as CommutingGateRouterAllToAll


from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples



In [2]:
scores = {i: np.exp(-1j*i) for i in np.linspace(-100,100, 1600+1)}
def get_score(x: complex):
    for i in np.linspace(-100,100, 1600+1):
        if np.abs(scores[i]- x) < 1e-4:
            return i 
    return x

In [51]:
def get_cost_circuit(filename, copy_numbers):
    filepath = f'/nfs/users/nfs_j/jc59/quantumwork/pangenome/data/{filename}.gfa'
    graph, n, V, T = gfa_file_to_graph(filepath, copy_numbers)
    normalised_hamiltonian, norm = graph_to_hubo_hamiltonian(graph, n, T, lamda=10, constraint_terms=1.0)
    hamiltonian = normalised_hamiltonian * norm


    extended_swap_strat = get_swap_strategy('all', n, T)
    num_physical_qubits = extended_swap_strat._num_vertices

    donor_qc = QuantumCircuit(num_physical_qubits)
    pm_rzz = PassManager(
        [
            FindCommutingPauliEvolutionsMulti(), 
            CommutingGateRouterPrecomputeRzz(
                extended_swap_strat,
                max_layers=0,
                perform_extra_swaps=True
            ),
            SwapToFinalMapping(),
            InverseCancellation(gates_to_cancel=[CXGate()]),
            CommutativeCancellation(basis_gates=["cx", "swap", "rz", "rzz"]),
            InverseCancellation(gates_to_cancel=[CXGate()]),
        ]
    )
    layout = Layout({donor_qc.qubits[i]: i for i in range(num_physical_qubits)})
    
    num_qubits: int = hamiltonian.num_qubits    

    keys = list(genbin(num_qubits))
    evals = evaluate_sparse_pauli_samples(keys, hamiltonian)
    opt_evals = np.nonzero(evals == 0)
    print((opt_evals[0]))
    circuit_hamiltonian = hamiltonian

    qc = QuantumCircuit(num_physical_qubits)
    qc.append(PauliEvolutionGate(circuit_hamiltonian), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_physical_qubits)])  
    cost_circuit = pm_rzz.run(qc)   
    cost_circuit = parameterise_circuit(cost_circuit, parameter=Parameter('γ'))
    return cost_circuit, circuit_hamiltonian, evals, layout, donor_qc, num_qubits


In [26]:
sim = AerSimulator()


In [47]:
def show_results(xs: list[int], hs: list[int], opt_key):
    p = 2
    delta_b = 0.5
    delta_g = 1
    # TRIVIAL WAS GOING TO MAX
    # HAVE TO SET GAMMA -> -GAMMA
    # WHY??
    
    x = np.array([(j-0.5)/p for j in range(1, p+1)])
    betas = delta_b * (1-x)
    gammas = delta_g * x
    print(betas, gammas)
    print()
    
    for sign in [-1, 1]:
        print('---------------')
        print(f'sign: {sign}')
        print('---------------')
        bqc = QuantumCircuit(cost_circuit.num_qubits)
        if len(xs):
            bqc.x(xs)
        if len(hs):
            bqc.h(hs)
            
        bind_dict = {cost_circuit.parameters[0]: gammas[0]}
        bound_cost_layer = cost_circuit.assign_parameters(bind_dict)
        bqc.compose(bound_cost_layer, range(cost_circuit.num_qubits), inplace=True)
        bqc.rx(sign * 2 * betas[0], range(num_qubits))
        
        bind_dict = {cost_circuit.parameters[0]: gammas[1]}
        bound_cost_layer = cost_circuit.assign_parameters(bind_dict)
        bqc.compose(bound_cost_layer, range(cost_circuit.num_qubits), inplace=True)
        bqc.rx(sign * 2 * betas[1], range(num_qubits))
        
        bqc.save_statevector()
        
        bqc2 = QuantumCircuit(cost_circuit.num_qubits)
        if len(xs):
            bqc2.x(xs)
        if len(hs):
            bqc2.h(hs)
        bqc2.append(PauliEvolutionGate(circuit_hamiltonian, gammas[0]), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_qubits)]) 
        bqc2.rx(sign * 2 * betas[0], range(num_qubits)) 
        
        bqc2.append(PauliEvolutionGate(circuit_hamiltonian, gammas[1]), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_qubits)]) 
        bqc2.rx(sign * 2 * betas[1], range(num_qubits)) 
        
        bqc2.save_statevector()
        tbqc2 = transpile(bqc2, sim)
        
        res = sim.run(bqc).result()
        sv = np.asarray(res.get_statevector())
        key = np.nonzero(np.abs(sv)**2 > sorted(np.abs(sv)**2)[-10] + 0.00001)[0]
        amp = sv[key]
        opt_amp = sv[opt_key]
        
        print('Compilation')
        # print(
        #     key, 
        #     evals[key], 
        #     amp / np.abs(amp) * np.exp(-1j * np.sum(gammas) * circuit_hamiltonian.coeffs[0]), 
        #     np.abs(amp) ** 2
        # )
        # print()
        print(
            opt_key,
            opt_amp / np.abs(opt_amp) * np.exp(-1j * np.sum(gammas) * circuit_hamiltonian.coeffs[0]), 
            np.abs(opt_amp * np.exp(-1j * circuit_hamiltonian.coeffs[0])) ** 2
        )
        print()
        
        # print('True')
        # print(key, evals[key], np.exp(-1j * evals[key]))
        
        res2 = sim.run(tbqc2).result()
        sv2 = np.asarray(res2.get_statevector())
        key2 = np.nonzero(np.abs(sv2)**2 > sorted(np.abs(sv2)**2)[-10] + 0.00001)[0]
        amp2 = sv2[key2]
        opt_amp2 = sv2[opt_key]
        print('No compilation')
        # print(key2, evals[key], amp2 / np.abs(amp2), np.abs(amp2) ** 2, )
        # print()
        print(opt_key, opt_amp2 / np.abs(opt_amp2),  np.abs(opt_amp2) ** 2)
        print()


In [48]:
cost_circuit, circuit_hamiltonian, evals, layout, donor_qc, num_qubits = get_cost_circuit('trivial', [1,1,1])
show_results([], range(9), np.array([17, 372]))

Keeping constraints at times: [1 0]
2
Max layers needed to apply swap decompose: 0
Applying phase 20.625
Gates we cannot directly implement: 0
Transpiling un-implemented gates
[0.375 0.125] [0.25 0.75]

---------------
sign: -1
---------------
Compilation
[ 17 372] [0.26569008-0.96405849j 0.26569008-0.96405849j] [0.02678871 0.02678871]

No compilation
[ 17 372] [0.26569008-0.96405849j 0.26569008-0.96405849j] [0.02678871 0.02678871]

---------------
sign: 1
---------------
Compilation
[ 17 372] [-0.50819482+0.86124214j -0.50819482+0.86124214j] [0.00219591 0.00219591]

No compilation
[ 17 372] [-0.50819482+0.86124214j -0.50819482+0.86124214j] [0.00219591 0.00219591]



In [49]:
cost_circuit, circuit_hamiltonian, evals, layout, donor_qc, num_qubits = get_cost_circuit('test_N2_W2', [1,1])
show_results([], range(4), np.array([1, 14]))

Keeping constraints at times: [0]
2
Max layers needed to apply swap decompose: 0
Applying phase 9.75
Gates we cannot directly implement: 0
Transpiling un-implemented gates
[0.375 0.125] [0.25 0.75]

---------------
sign: -1
---------------
Compilation
[ 1 14] [0.90240046-0.43089837j 0.90240046-0.43089837j] [0.26052447 0.26052447]

No compilation
[ 1 14] [0.90240046-0.43089837j 0.90240046-0.43089837j] [0.26052447 0.26052447]

---------------
sign: 1
---------------
Compilation
[ 1 14] [0.70324828+0.71094435j 0.70324828+0.71094435j] [0.09874519 0.09874519]

No compilation
[ 1 14] [0.70324828+0.71094435j 0.70324828+0.71094435j] [0.09874519 0.09874519]



In [54]:
cost_circuit, circuit_hamiltonian, evals, layout, donor_qc, num_qubits = get_cost_circuit('test_N7_W2', [1,0,0,0,0,0,1])
show_results([], range(8), np.array([3, 184]))
# 1011 1000

Keeping constraints at times: [0]
[  3 184]
Max layers needed to apply swap decompose: 0
Applying phase 12.109375
Gates we cannot directly implement: 0
Transpiling un-implemented gates
[0.375 0.125] [0.25 0.75]

---------------
sign: -1
---------------
Compilation
[  3 184] [0.98478944+0.17375202j 0.98478944+0.17375202j] [0.04696549 0.04696549]

No compilation
[  3 184] [0.98478944+0.17375202j 0.98478944+0.17375202j] [0.04696549 0.04696549]

---------------
sign: 1
---------------
Compilation
[  3 184] [0.77364305+0.63362168j 0.77364305+0.63362168j] [0.00440731 0.00440731]

No compilation
[  3 184] [0.77364305+0.63362168j 0.77364305+0.63362168j] [0.00440731 0.00440731]

